## 1. Environment Setup

In [14]:

%pip install -U pypdf python-dotenv sentence-transformers pinecone langchain-core langchain-community langchain-text-splitters langchain-pinecone langchain-huggingface


  Using cached pinecone-10.3.0-cp310-abi3-win_amd64.whl.metadata (7.0 kB)
Note: you may need to restart the kernel to use updated packages.


## 2. Imports and Configuration

In [15]:

import os
import hashlib
from pathlib import Path

from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

PROJECT_DIR = Path.cwd()
DOCUMENTS_DIR = PROJECT_DIR / "RAG" / "documents"

load_dotenv(PROJECT_DIR / ".env")

INDEX_NAME = "heart-disease-knowledge"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 5

NAMESPACE = ""

print("Project directory:", PROJECT_DIR)
print("Documents directory:", DOCUMENTS_DIR)
print("Pinecone index:", INDEX_NAME)


Project directory: C:\Users\Windows 10 Pro\PycharmProjects\CDSS_Project
Documents directory: C:\Users\Windows 10 Pro\PycharmProjects\CDSS_Project\RAG\documents
Pinecone index: heart-disease-knowledge


## 3. Medical Document Discovery

Locate the medical PDF documents without modifying them.

In [17]:
if not DOCUMENTS_DIR.is_dir():
    raise FileNotFoundError(
        f"Documents directory not found: {DOCUMENTS_DIR}"
    )

pdf_files = sorted(DOCUMENTS_DIR.glob("*.pdf"))

print("Number of PDF files:", len(pdf_files))

for pdf_file in pdf_files:
    print(pdf_file.name)

if not pdf_files:
    raise ValueError("No PDF documents were found.")


Number of PDF files: 31
Advancing Heart, Lung, Blood and Sleep Research.pdf
Blood Cholesterol - Causes and Risk Factors _ NHLBI, NIH.pdf
Blood Cholesterol - Diagnosis _ NHLBI, NIH.pdf
Blood Cholesterol - Living With High Cholesterol _ NHLBI, NIH.pdf
Blood Cholesterol - Treatment _ NHLBI, NIH.pdf
Blood Cholesterol - What is Blood Cholesterol_ _ NHLBI, NIH.pdf
Can you have a heart attack and not know it.pdf
Compendium of essential clinical terminology for hypertension-control programmes.pdf
Diabetes, Heart Disease, & Stroke - NIDDK.pdf
Guideline for the Management of Patients With Acute Coronary Syndromes.pdf
Guidelines for the management of cardiovascular disease and chronic kidney disease.pdf
Guidelines for the management of cardiovascular disease and pregnancy.pdf
Guidelines for the management of heart failure.pdf
Heart Attack - Causes and Risk Factors _ NHLBI, NIH.pdf
Heart Attack - Diagnosis _ NHLBI, NIH.pdf
Heart Attack - Heart Attacks in Women _ NHLBI, NIH.pdf
Heart Attack - Recov

## 4. PDF Text Extraction

Use LangChain's PyPDFLoader to extract individual pages while preserving source information.

In [18]:
documents = []
failed_files = []

for pdf_file in pdf_files:
    try:
        loader = PyPDFLoader(str(pdf_file))
        pages = loader.load()

        for page in pages:
            page.metadata["source"] = pdf_file.name
            page.metadata["page"] = (
                int(page.metadata.get("page", 0)) + 1
            )

            if page.page_content.strip():
                documents.append(page)

        print(f"Loaded {pdf_file.name}: "f"{len(pages)} pages")
    except Exception as error:
        failed_files.append((pdf_file.name, str(error)))
        print(f"FAILED: {pdf_file.name}: {error}")

print("Extracted nonempty pages:", len(documents))
print("Failed files:", len(failed_files))

if not documents:
    raise ValueError("No usable PDF text was extracted.")


Loaded Advancing Heart, Lung, Blood and Sleep Research.pdf: 19 pages
Loaded Blood Cholesterol - Causes and Risk Factors _ NHLBI, NIH.pdf: 3 pages
Loaded Blood Cholesterol - Diagnosis _ NHLBI, NIH.pdf: 5 pages
Loaded Blood Cholesterol - Living With High Cholesterol _ NHLBI, NIH.pdf: 5 pages
Loaded Blood Cholesterol - Treatment _ NHLBI, NIH.pdf: 5 pages
Loaded Blood Cholesterol - What is Blood Cholesterol_ _ NHLBI, NIH.pdf: 2 pages
Loaded Can you have a heart attack and not know it.pdf: 11 pages
Loaded Compendium of essential clinical terminology for hypertension-control programmes.pdf: 24 pages
Loaded Diabetes, Heart Disease, & Stroke - NIDDK.pdf: 9 pages
Loaded Guideline for the Management of Patients With Acute Coronary Syndromes.pdf: 92 pages
Loaded Guidelines for the management of cardiovascular disease and chronic kidney disease.pdf: 102 pages
Loaded Guidelines for the management of cardiovascular disease and pregnancy.pdf: 107 pages
Loaded Guidelines for the management of heart fa

## 5. Text Preprocessing and Chunking

Split extracted medical text into overlapping chunks.

In [19]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    add_start_index=True
)
chunks = text_splitter.split_documents(documents)
chunks = [
    chunk for chunk in chunks
    if chunk.page_content.strip()
]
print("Total chunks:", len(chunks))
if chunks:
    print("Source:", chunks[0].metadata["source"])
    print("Page:", chunks[0].metadata["page"])
    print("Preview:", chunks[0].page_content[:500])


Total chunks: 3872
Source: Advancing Heart, Lung, Blood and Sleep Research.pdf
Page: 1
Preview: ADVANCING
Heart, Lung, Blood, and Sleep Research
March 2025  |  WWW.NHLBI.NIH.GOV


## 6. Embedding Model

Load the same 384-dimensional embedding model used by the original Pinecone implementation.

In [20]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

test_embedding = embeddings.embed_query(
    "High blood pressure is a risk factor for heart disease."
)

print("Embedding dimensions:", len(test_embedding))

assert len(test_embedding) == 384


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimensions: 384


## 7. Pinecone Connection

Connect to the existing Pinecone account and index.

In [21]:
api_key = os.getenv("PINECONE_API_KEY")

if not api_key:
    raise ValueError("PINECONE_API_KEY is missing from .env")

pc = Pinecone(api_key=api_key)

if not pc.has_index(INDEX_NAME):
    raise ValueError( f"Index '{INDEX_NAME}' does not exist. "
        "Verify the Pinecone account and index name."
    )
index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()

print("Connected index:", INDEX_NAME)
print("Index dimensions:", stats.dimension)
print("Total vectors:", stats.total_vector_count)

assert stats.dimension == 384, ("Pinecone dimension does not match the embedding model."
)

Connected index: heart-disease-knowledge
Index dimensions: 384
Total vectors: 9690


## 8. LangChain Pinecone Vector Store

Connect LangChain to the existing vector index.

In [22]:

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    text_key="text",
    namespace=NAMESPACE
)

print("LangChain Pinecone vector store initialized.")


LangChain Pinecone vector store initialized.


## 9. Optional Document Indexing

Only run the upload cell when deliberately indexing new or updated documents. The existing index may already contain your medical knowledge.

In [23]:

def generate_chunk_id(document):
    content = (
        f"{document.metadata.get('source', '')}|"
        f"{document.metadata.get('page', '')}|"
        f"{document.metadata.get('start_index', '')}|"
        f"{document.page_content}"
    )

    return hashlib.sha256(
        content.encode("utf-8")
    ).hexdigest()


chunk_ids = [
    generate_chunk_id(chunk)
    for chunk in chunks
]

print("Chunks prepared:", len(chunk_ids))
print("Unique IDs:", len(set(chunk_ids)))

assert len(chunk_ids) == len(set(chunk_ids))


Chunks prepared: 3872
Unique IDs: 3872


In [24]:
ENABLE_INDEXING = False

if ENABLE_INDEXING:
    if failed_files:
        raise RuntimeError(
            "Resolve PDF loading failures before indexing."
        )
    batch_size = 32

    for start in range(0, len(chunks), batch_size):
        batch = chunks[start:start + batch_size]
        ids = chunk_ids[start:start + batch_size]

        vector_store.add_documents(
            documents=batch,
            ids=ids
        )

        print(f"Indexed {min(start + batch_size, len(chunks))}"
            f"/{len(chunks)}"
        )
    print("Document indexing completed.")
else:
    print("Indexing skipped. Existing vectors preserved.")

Indexing skipped. Existing vectors preserved.


## 10. LangChain Medical Evidence Retrieval

In [25]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)
def retrieve_medical_evidence(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("A nonempty question is required.")

    retrieved_documents = retriever.invoke(question)

    evidence = []

    for rank, document in enumerate(
        retrieved_documents, start=1
    ):
        page = document.metadata.get("page")

        if isinstance(page, (int, float)):
            if float(page).is_integer():
                page = int(page)

        evidence.append({
            "rank": rank,
            "source": document.metadata.get("source", "Unknown"),
            "page": page,
            "text": document.page_content
        })
    return evidence
print("LangChain retriever initialized.")


LangChain retriever initialized.


## 11. Retrieval Testing

In [26]:

question = (
    "What are the common risk factors for heart disease?"
)

results = retrieve_medical_evidence(question)

print("Question:", question)
print("Retrieved passages:", len(results))

for result in results:
    print("-" * 70)
    print("Rank:", result["rank"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Text:", result["text"][:700])


Question: What are the common risk factors for heart disease?
Retrieved passages: 5
----------------------------------------------------------------------
Rank: 1
Source: Promises for Healthy Hearts How Hispanics and Latinos Can Prevent Heart Disease.pdf
Page: 11
Text: Section 1
11
Promises for Healthy Hearts
Carmen
The doctor told me that I 
have high blood pressure and 
diabetes. Both put me at risk for 
heart disease. Now that I know, 
I’m taking steps to lower my risk.
Controlling Your Risks
Risk factors are traits and habits that make you more likely to develop 
heart disease. Some risk factors you can do something about; others you 
can’t change. The more risk factors you have, the greater your chances of 
developing heart disease.
----------------------------------------------------------------------
Rank: 2
Source: Heart attack - WHO.pdf
Page: 3
Text: A relatively small set of risk factors accounts for most of this process. Behavioural risk
factors include tobacco use, a diet h

## 12. BioMistral Integration Preparation

In [27]:

def format_evidence_context(evidence):
    sections = []

    for item in evidence:
        sections.append(
            f"Source: {item['source']}\n"
            f"Page: {item['page']}\n"
            f"Evidence:\n{item['text']}"
        )

    return "\n\n---\n\n".join(sections)


medical_context = format_evidence_context(results)

print("Context length:", len(medical_context))
print("Context preview:")
print(medical_context[:1500])


Context length: 4109
Context preview:
Source: Promises for Healthy Hearts How Hispanics and Latinos Can Prevent Heart Disease.pdf
Page: 11
Evidence:
Section 1
11
Promises for Healthy Hearts
Carmen
The doctor told me that I 
have high blood pressure and 
diabetes. Both put me at risk for 
heart disease. Now that I know, 
I’m taking steps to lower my risk.
Controlling Your Risks
Risk factors are traits and habits that make you more likely to develop 
heart disease. Some risk factors you can do something about; others you 
can’t change. The more risk factors you have, the greater your chances of 
developing heart disease.

---

Source: Heart attack - WHO.pdf
Page: 3
Evidence:
A relatively small set of risk factors accounts for most of this process. Behavioural risk
factors include tobacco use, a diet high in salt, sugar and unhealthy fats, physical inactivity,
and harmful use of alcohol. Air pollution is an important environmental contributor. Over
time, these behaviours tend to produce a